<a href="https://colab.research.google.com/github/broadinstitute/missense-pfes/blob/main/Copy_of_Compute_PFES.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
if 'google.colab' in sys.modules:
  %pip install g2papi
import g2papi


In [17]:
# @title # Input your gene/protein (HGNC symbol/UniProt accession) and a variant (e.g. M1V)
# @markdown Forms support many types of fields.

gene = 'UMOD'  # @param {type: "string"}
uniprot = 'P07911'  # @param {type: "string"}
variant = 'C77Y'  # @param {type: "string"}

## Import protein features from Genomics 2 Proteins portal via g2p

In [18]:
# Get protein features as a pandas dataframe
protein_features = g2papi.get_protein_features(gene, uniprot)
protein_features.fillna('-', inplace=True)

protein_features

/tmp/ipykernel_3484586/125996553.py:3: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '-' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  protein_features.fillna('-', inplace=True)


,residueId,AA,Amino acid residues,Amino acid properties,Secondary structure (PDBe/SIFTS),Secondary structure (DSSP 3-state)*,Secondary structure (DSSP 9-state)*,Accessible surface area (Å²)*,Phi angle (degrees)*,Psi angle (degrees)*,...,Intra-chain Non-bonded interaction (PDB),Intra-chain Non-bonded interaction (AlphaFold2),Intra-chain Disulfide bond (PDB),Intra-chain Disulfide bond (AlphaFold2),Intra-chain Salt bridge (PDB),Intra-chain Salt bridge (AlphaFold2),Inter-chain Hydrogen bond (PDB),Inter-chain Non-bonded interaction (PDB),Inter-chain Disulfide bond (PDB),Inter-chain Salt bridge (PDB)
0,1,M,Methionine,Aliphatic,-,C (loop/coil),C (loop/coil),254,360.0,119.5,...,-,-,-,-,-,-,-,-,-,-
1,2,G,Glycine,"Special, lack of a chiral carbon, smallest ami...",-,C (loop/coil),C (loop/coil),79,-150.8,169.3,...,-,-,-,-,-,-,-,-,-,-
2,3,Q,Glutamine,Polar/Neutral,-,C (loop/coil),C (loop/coil),190,-81.1,164.4,...,-,-,-,-,-,-,-,-,-,-
3,4,P,Proline,"Special, No backbone hydrogen",-,C (loop/coil),C (loop/coil),123,-91.6,172.7,...,-,-,-,-,-,-,-,-,-,-
4,5,S,Serine,Polar/Neutral,-,C (loop/coil),C (loop/coil),118,-151.9,135.2,...,-,-,-,-,-,-,-,-,-,-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
635,636,T,Threonine,Polar/Neutral,-,C (loop/coil),C (loop/coil),133,-92.0,123.9,...,-,-,-,-,-,-,-,-,-,-
636,637,L,Leucine,Aliphatic,-,C (loop/coil),C (loop/coil),141,-119.7,102.6,...,-,-,-,-,-,-,-,-,-,-
637,638,T,Threonine,Polar/Neutral,-,C (loop/coil),C (loop/coil),130,-98.4,123.2,...,-,-,-,-,-,-,-,-,-,-
638,639,F,Phenylalanine,Aromatic,-,C (loop/coil),C (loop/coil),184,-143.9,129.5,...,-,-,-,-,-,-,-,-,-,-


## Retreive PANTHER protein class from G2P metadata

In [27]:
import requests
import pandas as pd
import numpy as np
from io import StringIO

# Construct the public URL for the Google Cloud Storage object
bucket_name = 'g2p-portal'
file_path = 'portal_data/2026_q1_data/uniprot_metadata.tsv'
gcs_public_url = f'https://storage.googleapis.com/{bucket_name}/{file_path}'

try:
    response = requests.get(gcs_public_url)
    response.raise_for_status()  # Raise an exception for HTTP errors (4xx or 5xx)

    file_content = response.text

    # Load the file_content into a pandas DataFrame
    df_uniprot_metadata = pd.read_csv(StringIO(file_content), sep='\t')

    # Extract UniProt ID and PANTHER_protein_class
    uniprot_panther_data = df_uniprot_metadata[['UniprotKB_Entry', 'PANTHER_protein_class']]

    # Filter the uniprot_panther_data DataFrame using the uid (assuming 'uid' is defined)
    panther_class_for_uid = uniprot_panther_data[uniprot_panther_data['UniprotKB_Entry'] == uniprot]['PANTHER_protein_class']

    # Check if a class was found and print it
    if not panther_class_for_uid.empty:
        print(f"PANTHER Protein Class for UniProt ID '{uniprot}':")
        protein_class = panther_class_for_uid.iloc[0]
        display(protein_class)
        
    else:
        print(f"No PANTHER Protein Class found for UniProt ID '{uniprot}'.")

except requests.exceptions.RequestException as e:
    print(f"Error accessing GCS file: {e}")
    print("This might be due to the file/bucket not being publicly accessible or the path being incorrect.")

PANTHER Protein Class for UniProt ID 'P07911':


'transmembrane signal receptor'

## Preprocess annotation file and binning 


In [53]:
protein_features

,residueId,AA,Amino acid residues,Amino acid properties,Secondary structure (PDBe/SIFTS),Secondary structure (DSSP 3-state)*,Secondary structure (DSSP 9-state)*,Accessible surface area (Å²)*,Phi angle (degrees)*,Psi angle (degrees)*,...,Intra-chain Non-bonded interaction (PDB),Intra-chain Non-bonded interaction (AlphaFold2),Intra-chain Disulfide bond (PDB),Intra-chain Disulfide bond (AlphaFold2),Intra-chain Salt bridge (PDB),Intra-chain Salt bridge (AlphaFold2),Inter-chain Hydrogen bond (PDB),Inter-chain Non-bonded interaction (PDB),Inter-chain Disulfide bond (PDB),Inter-chain Salt bridge (PDB)
0,1,M,Methionine,Aliphatic,-,C (loop/coil),C (loop/coil),254,360.0,119.5,...,-,-,-,-,-,-,-,-,-,-
1,2,G,Glycine,"Special, lack of a chiral carbon, smallest ami...",-,C (loop/coil),C (loop/coil),79,-150.8,169.3,...,-,-,-,-,-,-,-,-,-,-
2,3,Q,Glutamine,Polar/Neutral,-,C (loop/coil),C (loop/coil),190,-81.1,164.4,...,-,-,-,-,-,-,-,-,-,-
3,4,P,Proline,"Special, No backbone hydrogen",-,C (loop/coil),C (loop/coil),123,-91.6,172.7,...,-,-,-,-,-,-,-,-,-,-
4,5,S,Serine,Polar/Neutral,-,C (loop/coil),C (loop/coil),118,-151.9,135.2,...,-,-,-,-,-,-,-,-,-,-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
635,636,T,Threonine,Polar/Neutral,-,C (loop/coil),C (loop/coil),133,-92.0,123.9,...,-,-,-,-,-,-,-,-,-,-
636,637,L,Leucine,Aliphatic,-,C (loop/coil),C (loop/coil),141,-119.7,102.6,...,-,-,-,-,-,-,-,-,-,-
637,638,T,Threonine,Polar/Neutral,-,C (loop/coil),C (loop/coil),130,-98.4,123.2,...,-,-,-,-,-,-,-,-,-,-
638,639,F,Phenylalanine,Aromatic,-,C (loop/coil),C (loop/coil),184,-143.9,129.5,...,-,-,-,-,-,-,-,-,-,-


# Retrieve Odd ratio data 

In [52]:
import requests
import pandas as pd
from io import StringIO

# Raw URL for the CSV file on GitHub
github_csv_url = 'https://raw.githubusercontent.com/broadinstitute/missense-pfes/refs/heads/main/results/enrichment_OR_by_protein_class.csv'
try:
    response = requests.get(github_csv_url)
    response.raise_for_status() # Raise an exception for HTTP errors (4xx or 5xx)

    # Read the content into a pandas DataFrame
    enrichment_df = pd.read_csv(StringIO(response.text), header=[0, 1], index_col=0)

    print("Successfully loaded the enrichment data. Here's a preview:")
    display(enrichment_df.head())

    # --- Extract odd ratio for a given protein class ---
    # Filter the DataFrame for the desired protein class
    odd_ratio_data = enrichment_df[protein_class][['OR','q_value']]


    # --- Handling infinite and zero values in odds ratios to stabilize log transformation --- 
    max_val = odd_ratio_data.loc[np.isfinite(odd_ratio_data['OR']), 'OR'].max()
    min_val = odd_ratio_data.loc[odd_ratio_data['OR'] > 0, 'OR'].min()
    odd_ratio_data['OR'] = np.where(odd_ratio_data['OR'] == np.inf, max_val, odd_ratio_data['OR'])
    odd_ratio_data['OR'] = np.where(odd_ratio_data['OR'] == 0, min_val, odd_ratio_data['OR'])

    if not odd_ratio_data.empty:
        print(f"\nOdd Ratio for '{protein_class}':")
        # Assuming 'Odd Ratio' is the column name for the odd ratio
        display(odd_ratio_data)
    else:
        print(f"\nNo data found for protein class: '{protein_class}'.")
        print("Please check the exact spelling of the protein class.")

except requests.exceptions.RequestException as e:
    print(f"Error accessing the GitHub CSV file: {e}")
    print("Please ensure the URL is correct and the file is publicly accessible.")

Successfully loaded the enrichment data. Here's a preview:


All                                                             \
               OR     CI_lo     CI_up       p_value       q_value n_case_yes   
feature                                                                        
SS:B     2.095935  1.863285  2.357634  1.042035e-35  1.579859e-35      688.0   
SS:E     1.949960  1.896735  2.004679  0.000000e+00  0.000000e+00    13002.0   
SS:G     1.353493  1.278055  1.433383  6.546386e-25  9.049416e-25     2331.0   
SS:H     1.673580  1.640216  1.707623  0.000000e+00  0.000000e+00    27537.0   
SS:I     3.117067  2.723904  3.566979  7.270310e-67  1.340018e-66      674.0   

                   DNA metabolism protein                      ...  \
        n_ctrl_yes                     OR     CI_lo     CI_up  ...   
feature                                                        ...   
SS:B         470.0               1.198662  0.542820  2.646901  ...   
SS:E       10362.0               2.320431  1.925335  2.796603  ...   
SS:G        2474.0               0.924600  0.634905  1.346477  ...   
SS:H       27431.0               1.788521  1.563988  2.045289  ...   
SS:I         310.0               1.055290  0.450050  2.474476  ...   

           transporter                       unclassified                      \
               q_value n_case_yes n_ctrl_yes           OR     CI_lo     CI_up   
feature                                                                         
SS:B      9.680528e-02       81.0       40.0     2.152166  1.613920  2.869919   
SS:E      1.006876e-02      830.0      499.0     2.737234  2.566682  2.919119   
SS:G      2.615007e-08      468.0      207.0     1.381684  1.198402  1.592996   
SS:H     2.199726e-182     6984.0     3162.0     1.746341  1.665998  1.830557   
SS:I      3.059482e-15      236.0       56.0     4.263885  3.041343  5.977856   

                                                             
               p_value        q_value n_case_yes n_ctrl_yes  
feature                                                      
SS:B      4.300968e-07   6.317047e-07       81.0      110.0  
SS:E     2.270489e-197  1.778550e-196     1971.0     2372.0  
SS:G      1.354697e-05   1.793542e-05      286.0      607.0  
SS:H     1.804233e-116  7.373823e-116     3866.0     7614.0  
SS:I      3.338566e-17   6.276504e-17       83.0       57.0  

[5 rows x 147 columns]


Odd Ratio for 'transmembrane signal receptor':


,OR,q_value
feature,,
SS:B,5.085862,4.316670e-11
SS:E,2.023126,6.172063e-44
SS:G,1.800144,3.494843e-07
SS:H,0.846824,5.898314e-05
SS:I,1.116114,7.261149e-01
...,...,...
Modification:Modified residue,3.475264,1.062683e-02
PPI:HB_inter,3.113775,9.769246e-23
PPI:SB_inter,2.564806,3.748816e-04
